# Visualización de una simulación TVB de una región

Este cuaderno carga uno o varios ficheros NumPy generados por los scripts de `src/zerlaut/` y compara sus resultados.

In [ ]:
%pwd

In [ ]:
result_dir = 'results'
RESULT_FILES = ['simple_simulation.npz']
# RESULT_FILES = ['simple_simulation.npz', 'simulation.npz']

# Se busca el fichero desde la raíz o desde un subdirectorio de notebooks.
from pathlib import Path
search_roots = [Path('../' * i) / result_dir for i in range(3)]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

results = {}
for result_file in RESULT_FILES:
    result_path = next(root / result_file for root in search_roots if (root / result_file).exists())
    label = Path(result_file).stem
    with np.load(result_path, allow_pickle=False) as stored:
        results[label] = {
            name: stored[name].copy() for name in stored.files
            if name != 'metadata_json'
        }

print(f'Ficheros cargados: {RESULT_FILES}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True, constrained_layout=True)

colors = plt.get_cmap('tab10').colors
for index, (label, result) in enumerate(results.items()):
    time_s = result['time_ms'] / 1000.0
    color = colors[index % len(colors)]
    axes[0].plot(time_s, result['stimulus_hz'], color=color, label=f'{label}: estímulo')
    axes[0].plot(time_s, result['Fe_ext_hz'], color=color, linestyle='--', label=f'{label}: Fe')
    axes[0].plot(time_s, result['Fi_ext_hz'], color=color, linestyle=':', label=f'{label}: Fi')
    axes[1].plot(time_s, result['E_hz'], color=color, label=f'{label}: E')
    axes[1].plot(time_s, result['I_hz'], color=color, linestyle='--', label=f'{label}: I')
    axes[2].plot(time_s, result['W_e_pA'], color=color, label=f'{label}: W_e')
    axes[2].plot(time_s, result['W_i_pA'], color=color, linestyle='--', label=f'{label}: W_i')

axes[0].set_ylabel('Entrada (Hz)')
axes[0].legend(loc='upper right')

axes[1].set_ylabel('Tasa (Hz)')
axes[1].legend(loc='upper right')

axes[2].set_ylabel('Adaptación (pA)')
axes[2].set_xlabel('Tiempo (s)')
axes[2].legend(loc='upper right')

for axis in axes:
    axis.grid(alpha=0.25)

fig.suptitle('Comparación de simulaciones Zerlaut-AdEx')
plt.show()

In [ ]:
summary = []
for label, result in results.items():
    summary.append({
        'fichero': label,
        'duración (s)': float(result['time_ms'][-1] / 1000.0),
        'E media (Hz)': float(np.mean(result['E_hz'])),
        'E máxima (Hz)': float(np.max(result['E_hz'])),
        'I media (Hz)': float(np.mean(result['I_hz'])),
        'I máxima (Hz)': float(np.max(result['I_hz'])),
    })
summary